# Lv3 Kaggle형 모의평가 2회

**예측 CSV 제출을 가정한 창작 문제입니다. 실제 Level 3의 제출물·평가 지표·리더보드는 미공개입니다.** 안내문 README.md의 규칙을 먼저 확인하세요. 독립적인 3문항이며 각 테스트 정답은 제공하지 않는 상황으로 가정합니다.

공통 규칙: 노트북은 모의문제 루트 또는 정답_채점용 폴더에서 실행합니다. 모델 선택에 test를 사용하지 않습니다. 문제별 제출 컬럼과 각 test ID는 샘플을 따릅니다.
\n**정답·베이스라인입니다.**

In [ ]:
from pathlib import Path
import sys, random, numpy as np, pandas as pd
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'practice_support.py').exists()),None)
if ROOT is None: raise FileNotFoundError('모의문제 폴더에서 실행하세요.')
sys.path.insert(0,str(ROOT.resolve()))
from practice_support import csv,images,save
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,log_loss,accuracy_score
random.seed(42); np.random.seed(42)
print('준비 완료')

## Q1 데이터 — 구매전환율 예측
`data/q1/train.csv`: id, price, ram, ssd, views, brand, target. `test.csv`: target 제외. 결측값이 있습니다.

1. 학습 데이터를 80/20으로 나누세요(seed 42). 결측치와 범주형을 전처리하고 회귀 모델을 학습하세요. 전처리는 training 부분에만 fit합니다.
2. validation의 RMSE를 출력하고 개선 후보를 비교해 보세요. 이 문항은 작은 값이 좋습니다.
3. 최종 선택 모델을 전체 train에 다시 학습하고 test 예측을 0~1로 제한하세요. sample_submission의 id 순서를 따르는 `id,prediction`을 `제출물/q1_submission.csv`로 저장하세요.
4. 예측 파일의 열, ID 중복·누락, 값 범위를 확인하세요.


In [ ]:
# 해설 답안
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
q1_train=csv('q1','train.csv'); q1_test=csv('q1','test.csv'); q1_sample=csv('q1','sample_submission.csv')
features=['price','ram','ssd','views','brand']; numerical=features[:-1]
prep=ColumnTransformer([('num',SimpleImputer(strategy='median'),numerical),('cat',OneHotEncoder(handle_unknown='ignore'),['brand'])])
model=Pipeline([('prep',prep),('model',RandomForestRegressor(n_estimators=80,min_samples_leaf=3,random_state=42,n_jobs=1))])
tr,va=train_test_split(np.arange(len(q1_train)),test_size=.2,random_state=42)
model.fit(q1_train.iloc[tr][features],q1_train.iloc[tr].target); p=np.clip(model.predict(q1_train.iloc[va][features]),0,1)
print('Q1 validation RMSE',mean_squared_error(q1_train.iloc[va].target,p)**.5)
model.fit(q1_train[features],q1_train.target)
result=pd.DataFrame({'id':q1_test.id,'prediction':np.clip(model.predict(q1_test[features]),0,1)})
q1_submit=q1_sample[['id']].merge(result,on='id',how='left',validate='one_to_one')
assert q1_submit.prediction.notna().all() and q1_submit.id.is_unique and q1_submit.prediction.between(0,1).all()
save(q1_submit,'q1_submission.csv')

## Q2 AI 모델 — 숫자 이미지 분류
`data/q2/train_images.npz`와 `test_images.npz`는 각각 id 배열과 `(N,8,8)` 이미지 배열을 가집니다. 픽셀은 0~16. `train_labels.csv`에는 id,label(0~9)이 있습니다. 행 순서를 가정하지 말고 id로 정답을 정렬하세요. test 레이블은 없습니다.

1. PyTorch 모델로 10개 클래스의 **logits**를 출력하세요. 학습에는 CrossEntropyLoss를 사용하고, 손실 계산 전에 softmax를 적용하지 않습니다.
2. 층화 80/20 validation을 만들고 학습 데이터의 손실과 validation log loss를 기록하세요. eval/no_grad에서 클래스 차원 softmax를 적용합니다. seed=42, CPU 사용.
3. sample_submission의 id 순서에 맞춰 `id,p0,...,p9` 확률 CSV를 저장하세요. 각 행의 확률 합이 1인지 확인합니다.
4. validation 결과로 모델을 선택하고 최종 예측 과정을 기록하세요. 제공하는 답안은 속도를 위한 베이스라인이며 더 나은 CNN이나 학습 계획을 시도할 수 있습니다.


In [ ]:
# 해설 답안: 8x8 이미지를 입력받는 소형 MLP. CNN으로 확장 가능
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
torch.manual_seed(42); torch.set_num_threads(2)
train_ids,train_images=images('train_images.npz'); test_ids,test_images=images('test_images.npz')
train_labels=csv('q2','train_labels.csv'); q2_sample=csv('q2','sample_submission.csv')
y=pd.DataFrame({'id':train_ids}).merge(train_labels,on='id',how='left',validate='one_to_one').label.to_numpy()
assert not pd.isna(y).any()
x=torch.tensor(train_images.reshape(-1,64)/16,dtype=torch.float32); y=torch.tensor(y,dtype=torch.long)
tr,va=train_test_split(np.arange(len(y)),test_size=.2,random_state=42,stratify=y.numpy())
model=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Linear(64,10)); opt=torch.optim.Adam(model.parameters(),lr=.003); criterion=nn.CrossEntropyLoss()
loader=DataLoader(TensorDataset(x[tr],y[tr]),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(42))
for epoch in range(15):
 model.train(); total=0
 for xb,yb in loader:
  opt.zero_grad(); logits=model(xb); loss=criterion(logits,yb); loss.backward(); opt.step(); total+=loss.item()*len(xb)
 model.eval()
 with torch.no_grad(): pv=torch.softmax(model(x[va]),dim=1).numpy()
 if epoch in [0,4,9,14]: print('epoch',epoch+1,'train loss',total/len(tr),'validation log loss',log_loss(y[va].numpy(),pv,labels=list(range(10))))
model.eval()
with torch.no_grad(): pt=torch.softmax(model(torch.tensor(test_images.reshape(-1,64)/16,dtype=torch.float32)),dim=1).numpy()
pt=pt.astype(np.float64); pt=pt/pt.sum(axis=1,keepdims=True)
probs=pd.DataFrame(pt,columns=[f'p{i}' for i in range(10)]); probs.insert(0,'id',test_ids)
q2_submit=q2_sample[['id']].merge(probs,on='id',how='left',validate='one_to_one')
assert q2_submit.id.is_unique and np.isfinite(q2_submit.iloc[:,1:].to_numpy()).all() and np.allclose(q2_submit.iloc[:,1:].sum(axis=1),1,atol=1e-5)
save(q2_submit,'q2_submission.csv')

## Q3 텍스트·LLM 활용 — 고객 문의 분류
`data/q3/train.csv`에는 id,text,label, test.csv에는 id,text가 있습니다. 가능한 레이블은 `fan`, `battery`, `upgrade`이며 평가지표는 accuracy입니다.

1. 층화 train/validation 분할 후 주어진 자원 내 텍스트 분류 모델을 만드세요. TF-IDF 베이스라인, 제공된 사전학습 모델 또는 LLM 프롬프트 방식을 사용할 수 있습니다. 이 연습 기본 파일에는 실제 LLM 가중치가 포함되지 않습니다.
2. validation 정확도와 클래스별 오류를 확인하고 모델을 개선해 보세요.
3. 샘플 ID 순서에 맞춰 `id,label` CSV를 만드세요. LLM 출력 사용 시 설명문에서 레이블을 안정적으로 추출하고, 알 수 없는 출력의 처리 방침을 기록하세요.
4. 실제 로컬 LLM 선택 실습과 제약은 Reference.md를 확인하세요.


In [ ]:
# 해설 답안: CPU TF-IDF 기반 베이스라인이며 LLM은 아님
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report
q3_train=csv('q3','train.csv'); q3_test=csv('q3','test.csv'); q3_sample=csv('q3','sample_submission.csv')
tr,va=train_test_split(np.arange(len(q3_train)),test_size=.2,random_state=42,stratify=q3_train.label)
model=make_pipeline(TfidfVectorizer(analyzer='char',ngram_range=(2,4)),LogisticRegression(max_iter=500,random_state=42))
model.fit(q3_train.iloc[tr].text,q3_train.iloc[tr].label); pred=model.predict(q3_train.iloc[va].text)
print('Q3 validation accuracy',accuracy_score(q3_train.iloc[va].label,pred));print(classification_report(q3_train.iloc[va].label,pred,zero_division=0))
model.fit(q3_train.text,q3_train.label); result=pd.DataFrame({'id':q3_test.id,'label':model.predict(q3_test.text)})
q3_submit=q3_sample[['id']].merge(result,on='id',how='left',validate='one_to_one')
assert q3_submit.id.is_unique and q3_submit.label.isin(['fan','battery','upgrade']).all()
save(q3_submit,'q3_submission.csv')

## 제출 전 확인\n세 파일의 ID, 열 순서, 행 수, 결측값, 값 범위를 검토하세요. 정답_채점용의 로컬 채점기는 연습용입니다.